# haluviz — fit a Jacobian lens for Qwen-1.8B-Chat on a Kaggle GPU

Before running: right panel → **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
Right panel → **Input → Upload → New Dataset**: upload `haluviz_colab.zip` and name the dataset `haluviz`.
Optionally also upload a Colab checkpoint (`...jlens_wikitext.pt` and `.pt.ckpt`) into the same dataset to resume.
Run the cells top to bottom. The lens ends up under **Output** (`/kaggle/working/lenses/`).

In [ ]:
import glob, os, shutil
zips = glob.glob('/kaggle/input/**/haluviz_colab.zip', recursive=True); assert zips, 'upload haluviz_colab.zip as a dataset first'
!rm -rf /kaggle/working/haluviz && mkdir -p /kaggle/working/haluviz /kaggle/working/lenses
!unzip -q -o {zips[0]} -d /kaggle/working/haluviz
for f in glob.glob('/kaggle/input/**/*jlens_wikitext.pt*', recursive=True):
    shutil.copy(f, '/kaggle/working/lenses/'); print('resuming from', f)
%cd /kaggle/working/haluviz
!pip install -q transformers gguf safetensors datasets huggingface_hub
import torch; print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Convert the GGUF into a native checkpoint (~3 min)
!python scripts/convert_gguf_to_hf.py --out /kaggle/working/model

In [ ]:
# Fit (resumes automatically if a checkpoint was copied above). ~1 min/prompt on a T4.
OUT = '/kaggle/working/lenses/qwen-1_8b-chat_jlens_wikitext.pt'
!python scripts/fit_lens.py --model-id /kaggle/working/model --dtype float32 \
    --n-prompts 30 --dim-batch 8 --max-seq-len 128 --out {OUT}
!ls -la /kaggle/working/lenses/

In [ ]:
# Sanity check: mid-layer readouts, J-lens vs logit lens
import sys; sys.path.insert(0, '/kaggle/working/haluviz')
from jspace_viz.model import load_model
from jspace_viz.lens import JacobianLens
from jspace_viz.analysis import read_grid
m = load_model('/kaggle/working/model', dtype='float16'); lens = JacobianLens.load(OUT); print(lens)
for mode in ('logit', 'jlens'):
    r = read_grid(m, lens, 'What is the capital city of Palau? Answer in one sentence.', mode=mode, chat=True, max_new_tokens=12, top_k=3)
    col = r['prompt_len'] - 1
    print(f"\n{mode}: answer={r['continuation']!r}")
    for row in r['grid'][::3]:
        print(f"  L{row['layer']:2d}: {[r['vocab'][i] for i in row['top_ids'][col]]}")
print('\nDone. Download the lens from the Output panel (lenses/).')